Named Entity Recognition (NER) 
• 	Entity Extraction: Identify people, organizations, locations, dates, products, etc.
• 	Enrichment: Add metadata to raw text for downstream tasks (retrieval, summarization, analytics).
• 	Agent Handoffs: A retriever agent can pass enriched text (with entities highlighted) to a summarizer or evaluator.

In [1]:
!pip install openai asyncio

In [2]:
from openai import AsyncOpenAI
import asyncio

client = AsyncOpenAI()

labels = [
    "person",      # people, including fictional characters
    "fac",         # buildings, airports, highways, bridges
    "org",         # organizations, companies, agencies, institutions
    "gpe",         # geopolitical entities like countries, cities, states
    "loc",         # non-gpe locations
    "product",     # vehicles, foods, appareal, appliances, software, toys 
    "event",       # named sports, scientific milestones, historical events
    "work_of_art", # titles of books, songs, movies
    "law",         # named laws, acts, or legislations
    "language",    # any named language
    "date",        # absolute or relative dates or periods
    "time",        # time units smaller than a day
    "percent",     # percentage (e.g., "twenty percent", "18%")
    "money",       # monetary values, including unit
    "quantity",    # measurements, e.g., weight or distance
]

class AsyncNERAgent:
    def __init__(self, model="gpt-4-turbo"):
        self.model = model
        self.instructions = instructions = (
            f"You are a Named Entity Recognition agent. "
            f"Extract entities from text and return them in JSON format "
            f"with categories: {', '.join(labels)}."
        )


    async def run(self, text):
        response = await client.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "system", "content": self.instructions},
                {"role": "user", "content": text}
            ]
        )
        return response.choices[0].message.content

# Example usage in Jupyter (top-level await works):
ner_agent = AsyncNERAgent()
result = await ner_agent.run("Satya Nadella announced a new Microsoft AI initiative in Redmond on March 24, 2026.")
print(result)

result = await ner_agent.run("In Germany, in 1440, goldsmith Johannes Gutenberg invented the movable-type printing press. His work led to an information revolution and the unprecedented mass-spread of literature throughout Europe. Modelled on the design of the existing screw presses, a single Renaissance movable-type printing press could produce up to 3,600 pages per workday.")
print(result)

{
  "person": ["Satya Nadella"],
  "org": ["Microsoft"],
  "gpe": ["Redmond"],
  "date": ["March 24, 2026"]
}
{
  "person": ["Johannes Gutenberg"],
  "org": [],
  "gpe": ["Germany", "Europe"],
  "loc": [],
  "product": ["movable-type printing press"],
  "event": ["information revolution"],
  "work_of_art": [],
  "law": [],
  "language": [],
  "date": ["1440"],
  "time": [],
  "percent": [],
  "money": [],
  "quantity": ["3,600 pages per workday"]
}
